
## Overview

This notebook will show you how to create and query a table or DataFrame that you uploaded to DBFS. [DBFS](https://docs.databricks.com/user-guide/dbfs-databricks-file-system.html) is a Databricks File System that allows you to store data for querying inside of Databricks. This notebook assumes that you have a file already inside of DBFS that you would like to read from.

This notebook is written in **Python** so the default cell type is Python. However, you can use different languages by using the `%LANGUAGE` syntax. Python, Scala, SQL, and R are all supported.

In [1]:
import findspark
findspark.init()pip install mysql-connector-python


In [2]:
import os
print(os.environ.get("HADOOP_HOME"))

C:\hadoop-3.0.0


In [3]:
from pyspark.sql import SparkSession

# Create SparkSession
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Jupyter PySpark") \
    .config("spark.executor.memory", "2g") \
    .getOrCreate()    

# Verify Spark Context
# print(spark.sparkContext.getConf().getAll())

In [ ]:
# files = dbutils.fs.ls("C:/Users/Dell/Documents/data_quality/data/error_data/data/Data_Quality_query-1.csv")
# for file in files:
#     dbutils.fs.rm(file.path, True)
# dbutils.fs.rm("C:/Users/Dell/Documents/data_quality/data/error_data/data/Data_Quality_query.csv", True)

In [19]:
import pyodbc

# Replace with your server name and database name
server = 'localhost'
database = 'master'
username = 'YourUsername'   # Optional if using Windows Authentication
password = 'YourPassword'   # Optional if using Windows Authentication

# Connection string
conn = pyodbc.connect(
    f'DRIVER={{ODBC Driver 17 for SQL Server}};'
    f'SERVER={server};'
    f'DATABASE={database};'
    f'Trusted_Connection=yes;'
)

cursor = conn.cursor()

# Test the connection
cursor.execute("SELECT * from dbo.table1")
print(cursor.fetchone())

# Close the connection
conn.close()

('sjwar', 21)


In [ ]:
import pyodbc
from pyspark.sql import SparkSession

# Initialize SparkSession
spark = SparkSession.builder \
    .appName("CSV to SQL Server") \
    .getOrCreate()

# Database connection settings
server = 'localhost'
database = 'master'
username = 'YourUsername'   # Optional if using Windows Authentication
password = 'YourPassword'   # Optional if using Windows Authentication

# Connection string
conn = pyodbc.connect(
    f'DRIVER={{ODBC Driver 17 for SQL Server}};'
    f'SERVER={server};'
    f'DATABASE={database};'
    f'Trusted_Connection=yes;'
)

cursor = conn.cursor()

# Define file paths
file_paths = [
    "C:/Users/Dell/Documents/data_quality/data/error_data/Customer.csv",
    "C:/Users/Dell/Documents/data_quality/data/error_data/CustomerTexts.csv",
    "C:/Users/Dell/Documents/data_quality/data/error_data/CustomerType.csv",
    "C:/Users/Dell/Documents/data_quality/data/error_data/CustomerTypeTexts.csv",
    "C:/Users/Dell/Documents/data_quality/data/error_data/CustomersHierarchy.csv",
    "C:/Users/Dell/Documents/data_quality/data/error_data/FinancialTransactions.csv",
    "C:/Users/Dell/Documents/data_quality/data/error_data/GLAccountType.csv",
    "C:/Users/Dell/Documents/data_quality/data/error_data/GLAccountTypeTexts.csv",
    "C:/Users/Dell/Documents/data_quality/data/error_data/GLAccountsHierarchy.csv",
    "C:/Users/Dell/Documents/data_quality/data/error_data/GLAccountsTexts.csv",
    "C:/Users/Dell/Documents/data_quality/data/error_data/GLAccounts.csv",
    "C:/Users/Dell/Documents/data_quality/data/error_data/ProductCategories.csv",
    "C:/Users/Dell/Documents/data_quality/data/error_data/ProductCategoryTexts.csv",
    "C:/Users/Dell/Documents/data_quality/data/error_data/ProductHierarchy.csv",
    "C:/Users/Dell/Documents/data_quality/data/error_data/ProductTexts.csv",
    "C:/Users/Dell/Documents/data_quality/data/error_data/Product.csv",
    "C:/Users/Dell/Documents/data_quality/data/error_data/ProfitCenter.csv",
    "C:/Users/Dell/Documents/data_quality/data/error_data/ProfitCenterHierarchy.csv",
    "C:/Users/Dell/Documents/data_quality/data/error_data/ProfitCenterTexts.csv"
]

# Loop through files and create tables
for file_path in file_paths:
    table_name = file_path.split("/")[-1].replace(".csv", "").lower()  # Extract filename as table name
    
    # Read CSV file using Spark
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(file_path)
    )
    
    # Create table statement dynamically based on DataFrame schema
    columns = ", ".join([f"[{field.name}] {field.dataType.simpleString().upper()}" for field in df.schema.fields])
    create_table_query = f"IF OBJECT_ID('dbo.{table_name}', 'U') IS NULL CREATE TABLE dbo.{table_name} ({columns})"
    
    # Execute the create table statement
    try:
        cursor.execute(create_table_query)
        conn.commit()
        print(f"✅ Table '{table_name}' created (if not exists).")
    except Exception as e:
        print(f"❌ Error creating table '{table_name}': {e}")

    # Write data into the table using pyodbc
    insert_query = f"INSERT INTO dbo.{table_name} VALUES ({', '.join(['?' for _ in df.columns])})"
    
    try:
        for row in df.collect():
            cursor.execute(insert_query, tuple(row))
        conn.commit()
        print(f"✅ Data inserted into '{table_name}'.")
    except Exception as e:
        print(f"❌ Error inserting into '{table_name}': {e}")

# Close the connection
conn.close()
spark.stop()


In [ ]:

from pyspark.sql import SparkSession

# Initialize SparkSession
spark = SparkSession.builder.enableHiveSupport().getOrCreate()

# Define file paths
file_paths = [
    "C:/Users/Dell/Documents/data_quality/data/error_data/Customer.csv",
    "C:/Users/Dell/Documents/data_quality/data/error_data/CustomerTexts.csv",
    "C:/Users/Dell/Documents/data_quality/data/error_data/CustomerType.csv",
    "C:/Users/Dell/Documents/data_quality/data/error_data/CustomerTypeTexts.csv",
    "C:/Users/Dell/Documents/data_quality/data/error_data/CustomersHierarchy.csv",
    "C:/Users/Dell/Documents/data_quality/data/error_data/FinancialTransactions.csv",
    "C:/Users/Dell/Documents/data_quality/data/error_data/GLAccountType.csv",
    "C:/Users/Dell/Documents/data_quality/data/error_data/GLAccountTypeTexts.csv",
    "C:/Users/Dell/Documents/data_quality/data/error_data/GLAccountsHierarchy.csv",
    "C:/Users/Dell/Documents/data_quality/data/error_data/GLAccountsTexts.csv",
    "C:/Users/Dell/Documents/data_quality/data/error_data/GLAccounts.csv",
    "C:/Users/Dell/Documents/data_quality/data/error_data/ProductCategories.csv",
    "C:/Users/Dell/Documents/data_quality/data/error_data/ProductCategoryTexts.csv",
    "C:/Users/Dell/Documents/data_quality/data/error_data/ProductHierarchy.csv",
    "C:/Users/Dell/Documents/data_quality/data/error_data/ProductTexts.csv",
    "C:/Users/Dell/Documents/data_quality/data/error_data/Product.csv",
    "C:/Users/Dell/Documents/data_quality/data/error_data/ProfitCenter.csv",
    "C:/Users/Dell/Documents/data_quality/data/error_data/ProfitCenterHierarchy.csv",
    "C:/Users/Dell/Documents/data_quality/data/error_data/ProfitCenterTexts.csv"
]

# Set database (Hive metastore)
database_name = "default"

# Ensure the database exists
spark.sql(f"CREATE DATABASE IF NOT EXISTS {database_name}")

# Loop through files and create tables
for file_path in file_paths:
    table_name = file_path.split("/")[-1].replace(".csv", "").lower()  # Extract filename as table name
    
    # Read CSV file
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(file_path)
    )
    
    # Register as a Hive table in the default metastore
    full_table_name = f"{database_name}.{table_name}"
    df.write.mode("overwrite").saveAsTable(full_table_name)
    
    print(f"✅ Registered table: {full_table_name}")

✅ Registered table: default.customer
✅ Registered table: default.customertexts
✅ Registered table: default.customertype
✅ Registered table: default.customertypetexts
✅ Registered table: default.customershierarchy
✅ Registered table: default.financialtransactions
✅ Registered table: default.glaccounttype
✅ Registered table: default.glaccounttypetexts
✅ Registered table: default.glaccountshierarchy
✅ Registered table: default.glaccountstexts
✅ Registered table: default.glaccounts
✅ Registered table: default.productcategories
✅ Registered table: default.productcategorytexts
✅ Registered table: default.producthierarchy
✅ Registered table: default.producttexts
✅ Registered table: default.product
✅ Registered table: default.profitcenter
✅ Registered table: default.profitcenterhierarchy
✅ Registered table: default.profitcentertexts


In [ ]:
from pyspark.sql import SparkSession

# Initialize SparkSession with Hive support
spark = SparkSession.builder.enableHiveSupport().getOrCreate()

# Define database name (Hive metastore)
database_name = "default"

# Ensure the database exists
spark.sql(f"CREATE DATABASE IF NOT EXISTS {database_name}")

# Get all CSV files dynamically from the folder
file_paths = [file.path for file in dbutils.fs.ls("C:/Users/Dell/Documents/data_quality/data/error_data/") if file.name.endswith(".csv")]

# Loop through files and create tables
for file_path in file_paths:
    table_name = file_path.split("/")[-1].replace(".csv", "").lower()  # Extract filename as table name
    
    # Read CSV file with inferred schema
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(file_path)
    )
    
    # Save as Hive Table
    full_table_name = f"{database_name}.{table_name}"
    df.write.mode("overwrite").saveAsTable(full_table_name)
    
    print(f"✅ Registered table: {full_table_name}")


✅ Registered table: default.data_quality_query
